In [ ]:
!pip install -q pypdf

In [ ]:
from pypdf import PdfReader
from google.colab import files

uploaded = files.upload()

In [ ]:
reader = PdfReader("Hybrid_Search_Practice.pdf")
rec_pdf = ""
for rec in reader.pages:
  rec_pdf += rec.extract_text()

In [ ]:
!pip install langchain-text-splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

chunk_list = text_splitter.split_text(rec_pdf)

In [ ]:
pip install -q sentence-transformers


In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = embedding_model.encode(chunk_list)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
!pip install -q accelerate faiss-cpu

import faiss
import numpy as np

embedding_dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(embedding_dimension)
index.add(embeddings)

In [ ]:
question = "what is mean by hybrid search?"
question_embedding = embedding_model.encode([question])

distance, index_number = index.search(
    np.array(question_embedding),
    k=5
)

retrieved_chunks = [chunk_list[rec] for rec in index_number[0]]

In [ ]:
!pip install -q scikit-learn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(chunk_list)

question_tfidf = vectorizer.transform([question])
similarity_scores = cosine_similarity(question_tfidf,X)
top_k = 5

top_indices = np.argsort(similarity_scores[0])[::-1][:top_k]
keyword_chunk = [chunk_list[rec] for rec in top_indices]

keyword_indices = top_indices.tolist()
semantic_indices = index_number[0].tolist()

print(keyword_indices,semantic_indices)
combined_indices = keyword_indices + semantic_indices

unique_list = list(dict.fromkeys(combined_indices))
print(unique_list)

unique_list_text = [chunk_list[rec] for rec in unique_list]

context_list = [[question,chunk_list[rec]] for rec in unique_list]
print(context_list)


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

chatbot = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)


In [ ]:
from sentence_transformers import CrossEncoder

# Load a pre-trained CrossEncoder model
model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
scores = model.predict(context_list)
print(scores)
sorted_idx = np.argsort(scores)[::-1]
print(sorted_idx)

best_cross_encoders_chunks = []

for idx in sorted_idx[:3]:
    best_cross_encoders_chunks.append(unique_list_text[idx])

context = "\n\n".join(best_cross_encoders_chunks)

In [ ]:
prompt = f"""
<|user|>

Use ONLY the context below.

Context:
{context}

Question:
{question}

If the answer is not present, reply exactly:

I couldn't find that information.

<|assistant|>
"""
response = chatbot(
    prompt,
    max_new_tokens=120,
    do_sample=False,
    return_full_text=False
)

answer = response[0]["generated_text"].strip()

print(answer)